# Player Shots Pricing Model

### English Football Pricing Model

This notebook presents a portfolio version of a pre-match player-shot pricing model developed using historical data from the Premier League, Championship, League One and League Two.

The objective is to estimate a player's expected number of shots before kick-off and convert that expectation into fair probabilities for common player-shot markets.

The modelling structure is:

**League Environment → Team Shot Expectation → Player Shot Allocation → Expected Minutes → Calibration → Market Probabilities**

The full production data pipeline and live pricing implementation are not included in this public repository. This notebook focuses on the modelling methodology, historical testing and selected results.

## 1. Modelling Problem & Data

### The Problem

Player-shot markets require two related questions to be answered before kick-off:

1. **How many shots is the player's team likely to take?**
2. **What proportion of those shots is the player likely to account for?**

A player's historical shots-per-game average alone does not account for the expected match environment.

For example, the same forward should not necessarily receive the same shot expectation when playing at home against a side that allows high shot volumes as when playing away against a strong defensive opponent.

The model therefore separates **team shot volume** from **player shot allocation**.

---

### Historical Database

The wider project contains **8,500+ completed matches** across:

- Premier League
- Championship
- League One
- League Two

The database covers multiple seasons and contains match, team and player-level information.

Relevant inputs for the player-shot model include:

- Match and competition
- Home/away status
- Team and opponent
- Team shots
- Player shots
- Player minutes
- Starting status
- Historical line-ups
- Recent player shot share
- Historical team shot creation
- Historical opponent shot concession

Only information available **before the match being predicted** is used when constructing pre-match features.

---

### Avoiding Future Information

A major requirement of the project is preventing information from the match being predicted from entering its own prediction.

Historical features are therefore shifted backwards before being used.

For example, a player's previous-10 shot share for Match 11 is calculated only from Matches 1–10.

This allows the model to be tested in a way that more closely represents how it would have operated live.

---

### Evaluation

Model development uses chronological testing rather than judging performance only on the same matches used to develop the model.

Earlier matches are used to develop or calibrate the model and later matches are retained as unseen data.

Performance is assessed using measures including:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Correlation
- Probability calibration
- Brier Score

The objective is not simply to fit historical results, but to produce probabilities that remain useful on future matches.

## 2. Building the Team Shot Model

Player shot expectations depend partly on how many shots the player's team is expected to generate.

The first stage of the player model is therefore to estimate **expected team shots**.

### Baseline

The starting benchmark was the historical pre-match league average for teams playing in the same home/away environment.

This provides a deliberately simple benchmark:

> **How many shots would an average team in this league be expected to take in this venue situation?**

A useful model should improve upon this benchmark on unseen matches.

---

### First Attempt — Recent Team and Opponent Averages

An initial model combined:

- The team's historical shot production
- The opponent's historical shots conceded

Longer season-to-date histories performed better than shorter five and ten-match windows.

The season model reduced MAE from approximately **3.84 shots for the league home/away baseline to 3.72**.

However, examining only the headline error metric exposed an important problem.

#### Home Advantage Was Being Lost

Actual historical shot averages were approximately:

| Venue | Actual Shots | Initial Model |
|---|---:|---:|
| Away | 11.11 | 12.32 |
| Home | 13.41 | 12.17 |

Although the model improved overall prediction error, it compressed both home and away teams towards approximately the same shot expectation.

This was structurally undesirable.

Home advantage is a major part of the match environment, and improving one error metric while removing that effect did not represent the football realistically.

---

### Redesign — Separate Team Strength From Match Environment

The model was redesigned around two separate concepts:

**1. Team strength**

How strongly does the team create shots, and how strongly does the opponent concede them, relative to the league?

**2. Match environment**

What shot volume would normally be expected for a home or away team in this league?

The resulting structure became:

**Expected Team Shots = League Home/Away Environment × Team Shot-Attack Strength × Opponent Shot-Concession Strength**

Team and opponent strengths are regressed towards league average to reduce the influence of noisy or extreme historical samples.

A chronological historical test selected approximately **60% weighting towards the observed team/opponent strength**, with the remaining weight pulled towards league average.

---

### Out-of-Sample Result

On later unseen matches:

| Model | MAE |
|---|---:|
| League Home/Away Baseline | 3.87 |
| Initial Season Model | 3.76 |
| Final Team Shot Model | **3.63** |

The final model therefore improved prediction error while also preserving the underlying home/away shot environment.

This became the team-shot component used by Player Shots V1.

In [1]:
def expected_team_shots(
    league_venue_environment,
    team_attack_strength,
    opponent_concession_strength
):
    """
    Simplified representation of the team-shot model.

    Strength values are calculated pre-match and
    regressed towards the league average.
    """

    return (
        league_venue_environment
        * team_attack_strength
        * opponent_concession_strength
    )


# Historical fake-live example:
# Liverpool v Brentford, May 2026

brentford_expected_shots = expected_team_shots(
    league_venue_environment=11.155,
    team_attack_strength=0.9128,
    opponent_concession_strength=0.9518
)

print(
    "Brentford expected shots:",
    round(brentford_expected_shots, 2)
)

Brentford expected shots: 9.69


## 3. Allocating Team Shots to Players

Once expected team-shot volume has been estimated, the next problem is allocating that volume between individual players.

### Previous-10 Shot Share

Rather than using a player's raw shots-per-game average, Player Shots V1 measures the proportion of his team's shots that the player accounted for across his previous ten appearances.

The calculation is:

**Previous-10 Shot Share = Player Shots / Team Shots**

using the same previous ten appearances.

This allows player expectations to respond to both individual shooting behaviour and the attacking environment in which those shots occurred.

For example, taking two shots in a match where the team produced eight is treated differently from taking two when the team produced twenty.

Historical testing found the previous-10 shot-share measure to be more useful than shorter five-match histories.

---

### Expected Minutes

Shot opportunity also depends on playing time.

Using actual match minutes would introduce future information, so the live model requires a pre-match estimate.

Historical testing showed that knowing the confirmed starting XI substantially improved this process.

For confirmed starters with sufficient history, expected minutes are based on their recent starting appearances.

On an unseen historical sample, using the previous five starts produced an expected-minutes MAE of approximately **7.74 minutes**, compared with **13.92 minutes** when using recent appearances without accounting for confirmed starting status.

Expected minutes are then compared with the playing time underlying the player's recent shot history.

---

### Player Shot Expectation

The resulting structure is:

**Raw Expected Player Shots = Expected Team Shots × Previous-10 Shot Share × Expected-Minutes Adjustment**

This deliberately keeps the first version of the model interpretable.

The team model determines the expected volume available.

The player's recent shot share determines how much of that volume he has historically accounted for.

Expected minutes then adjust that share for the amount of playing time expected today.

In [2]:
def raw_player_shot_expectation(
    expected_team_shots,
    previous_10_shot_share,
    expected_minutes,
    historical_average_minutes
):
    """
    Simplified representation of Player Shots V1.

    The production model includes additional validation,
    eligibility checks and calibration.
    """

    minutes_adjustment = (
        expected_minutes
        / historical_average_minutes
    )

    # Prevent extreme adjustments from very unusual
    # historical playing-time samples.
    minutes_adjustment = max(
        0.25,
        min(2.0, minutes_adjustment)
    )

    return (
        expected_team_shots
        * previous_10_shot_share
        * minutes_adjustment
    )


# Historical fake-live example:
# Dango Ouattara v Liverpool

ouattara_raw_shots = raw_player_shot_expectation(
    expected_team_shots=9.692,
    previous_10_shot_share=0.1875,
    expected_minutes=89.8,
    historical_average_minutes=72.8
)

print(
    "Ouattara raw expected shots:",
    round(ouattara_raw_shots, 2)
)

Ouattara raw expected shots: 2.24


## 4. Testing Football Ideas Rather Than Assuming They Work

A major part of model development is testing whether football ideas contain genuine predictive information.

### Opponent Positional Shot Concession

One hypothesis was that some teams may systematically allow more shots to particular positions.

For example:

> If an opponent has repeatedly conceded high shot volumes to right-backs, should today's right-back receive a higher shot expectation?

Historical starting positions were reconstructed from line-up and formation data, allowing opponent shot concession to be measured separately for positions including:

- Centre-forward
- Left and right wing
- Attacking midfield
- Central midfield
- Wing-back
- Full-back
- Centre-back

Opponent positional shot-concession rates were tested using previous 10, 25 and 50-match windows.

### Result

The feature did **not** improve prediction.

| Opponent History | Baseline MAE | Positional Model MAE | Improvement |
|---|---:|---:|---:|
| Previous 10 | 0.8978 | 0.9273 | -0.0295 |
| Previous 25 | 0.8995 | 0.9113 | -0.0117 |
| Previous 50 | 0.8996 | 0.9066 | -0.0070 |

All three versions increased prediction error.

The feature was therefore **rejected from Player Shots V1**.

### Why This Matters

A football explanation can sound convincing without being predictive.

A team may appear to concede unusually high shot volumes to a position simply because it recently faced players in that position who naturally shoot more often.

This suggests that a better future test would ask:

> **Do players shoot more against this opponent than we would have expected from their own pre-match shooting behaviour?**

That would measure whether the opponent creates an additional positional effect rather than simply counting the shots it previously conceded.

For V1, however, there was insufficient evidence to include an opponent positional adjustment.

## 5. Historical Backtesting & Calibration

A useful pricing model needs to do more than rank players correctly. Its expected-shot values also need to translate into realistic probabilities.

### Historical Backtest

Player Shots V1 was tested across a large historical sample using only information available before each match.

The initial backtest produced more than **200,000 player-match predictions** across approximately **7,800 matches**.

The model showed a clear relationship between predicted and actual player shot volume, but analysis of the predictions revealed an important calibration issue.

### Raw Expected Shots Were Too Extreme

Players given very low expected-shot values tended to shoot more often than the raw model predicted.

At the other end, players given very high expected-shot values tended to shoot less often than predicted.

In other words, the model was useful at distinguishing high-volume shooters from low-volume shooters, but the raw expectations were **too spread out**.

Rather than ignoring this problem, a calibration layer was estimated using older historical matches and tested on later unseen matches.

This pulls extreme predictions towards more realistic historical outcomes while preserving the underlying ranking between players.

---

### Chronological Testing

The data was divided chronologically:

- **Older 75%:** model calibration
- **Newest 25%:** unseen evaluation

This is preferable to a random train/test split for this application because the objective is to simulate predicting future football matches from past information.

The calibration parameters were therefore learned without using the later matches on which they were evaluated.

---

### From Expected Shots to Probabilities

After calibration, each player's expected shot count can be converted into probabilities for betting markets such as:

- 1+ shots
- 2+ shots
- 3+ shots
- 4+ shots
- 5+ shots

For V1, a Poisson distribution is used to make this conversion.

Alternative count distributions were also tested. A Negative Binomial model represented some parts of the observed shot distribution slightly better, particularly the greater frequency of zero-shot and high-shot performances.

However, improvements in probability accuracy were very small and inconsistent across thresholds.

Poisson was therefore retained for V1 because it provided a simpler and more interpretable pricing framework without a meaningful loss of predictive performance.

---

### Important Limitation

Player shots are not perfectly Poisson distributed.

Historical results showed slightly more zero-shot performances and more extreme high-shot performances than a Poisson model would expect.

This limitation is documented rather than hidden and remains an area for future model development.

In [3]:
import math


def probability_at_least_n_shots(expected_shots, n):
    """
    Convert expected shots into the probability
    of recording at least n shots using Poisson.
    """

    probability_below_n = sum(
        math.exp(-expected_shots)
        * expected_shots**k
        / math.factorial(k)
        for k in range(n)
    )

    return 1 - probability_below_n


# Example:
# A player with 1.50 expected shots

expected_shots = 1.50

for threshold in [1, 2, 3, 4, 5]:

    probability = probability_at_least_n_shots(
        expected_shots,
        threshold
    )

    fair_odds = 1 / probability

    print(
        f"{threshold}+ shots: "
        f"{probability:.1%} | "
        f"Fair odds: {fair_odds:.2f}"
    )

1+ shots: 77.7% | Fair odds: 1.29
2+ shots: 44.2% | Fair odds: 2.26
3+ shots: 19.1% | Fair odds: 5.23
4+ shots: 6.6% | Fair odds: 15.23
5+ shots: 1.9% | Fair odds: 53.83


## 6. Historical Fake-Live Case Study

### Liverpool v Brentford — 24 May 2026

To demonstrate how the model operates in practice, a completed historical match was recreated as if it had not yet been played.

The model was given information that would have been available before kick-off, including:

- Historical team performance
- Historical opponent performance
- Confirmed starting XI
- Previous player shot history
- Previous player minutes
- Home/away environment

No shots, minutes or other performance information from the match itself were used to produce the forecasts.

Predictions were frozen before the actual match data was revealed.

---

### Stage 1 — Expected Team Shots

The team-shot model produced:

**Brentford expected shots: 9.69**

The model combined the Premier League away-team shot environment with Brentford's pre-match shot-creation strength and Liverpool's pre-match shot-concession strength.

Brentford subsequently recorded:

**Actual shots: 11**

A single match is not evidence that the team model is accurate, but this provides a useful illustration of how the system operates before kick-off.

---

### Stage 2 — Player Expectations

Using the confirmed Brentford starting XI, the 9.69 expected team shots were allocated using each player's pre-match shooting history and expected playing time.

The highest player expectations were:

| Player | Expected Shots |
|---|---:|
| Dango Ouattara | 1.97 |
| Igor Thiago | 1.49 |
| Kevin Schade | 1.10 |
| Keane Lewis-Potter | 1.09 |
| Sepp van den Berg | 0.94 |
| Nathan Collins | 0.77 |
| Mathias Jensen | 0.70 |

These were model outputs frozen before revealing the players' actual shot totals.

---

### Stage 3 — Converting Expectations Into Prices

The expected-shot values were converted into market probabilities.

Selected pre-match model prices included:

| Player | 1+ | 2+ | 3+ |
|---|---:|---:|---:|
| Dango Ouattara | 86.0% | 58.5% | 31.4% |
| Igor Thiago | 77.4% | 43.8% | 18.8% |
| Kevin Schade | 66.6% | 30.0% | 9.9% |
| Keane Lewis-Potter | 66.5% | 29.9% | 9.8% |

These probabilities represent model-derived fair estimates rather than bookmaker prices.

### Stage 4 — Reveal

Only after the forecasts had been frozen were the actual player-shot totals revealed.

| Player | Expected Shots | Actual Shots |
|---|---:|---:|
| Dango Ouattara | 1.97 | 3 |
| Igor Thiago | 1.49 | 2 |
| Kevin Schade | 1.10 | 4 |
| Keane Lewis-Potter | 1.09 | 1 |
| Sepp van den Berg | 0.94 | 0 |
| Nathan Collins | 0.77 | 0 |
| Mathias Jensen | 0.70 | 1 |
| Vitaly Janelt | 0.57 | 0 |
| Michael Kayode | 0.44 | 0 |

The four players with the highest attacking shot expectations — Ouattara, Thiago, Schade and Lewis-Potter — accounted for **10 of Brentford's 11 shots**.

This result should not be interpreted as proof of model accuracy.

The purpose of the case study is to demonstrate the complete pre-match workflow on a match that was withheld until after the model outputs had been generated. Model quality is assessed using the much larger chronological historical backtest rather than whether individual selections won in this match.

---

### What the Test Exposed

The fake-live test also highlighted a limitation in the current calibration approach.

A global calibration intercept can assign a small positive shot expectation even when a player's raw expectation is zero. This is particularly inappropriate for goalkeepers.

Rather than changing the model after observing this match, the issue is documented as a future improvement.

This reflects the wider development philosophy of the project:

**Build → Backtest → Diagnose → Improve → Retest**

## 7. Limitations & Next Steps

Player Shots V1 is deliberately designed as an interpretable first version rather than a finished production model.

Several areas remain available for further testing.

### Expected Minutes

Confirmed starting line-ups substantially improve expected-minute estimates, but substitutions remain difficult to predict.

Future versions could incorporate:

- Player-specific substitution tendencies
- Team and manager substitution patterns
- Positional substitution behaviour
- Competition for minutes within the squad
- Match-state expectations

### Player Shooting History

Previous-10 shot share performed best among the initial historical windows tested, outperforming previous-five, season-to-date and career-to-date measures.

Future testing could explore additional windows and time-decay approaches rather than using a fixed ten-appearance cutoff.

### Tactical Matchups

Simple opponent positional shot-concession rates did not improve prediction and were rejected from V1.

A more sophisticated future approach could compare an opponent's positional shot concessions with the pre-match expectations of the players it previously faced.

This would test whether an opponent systematically causes players in particular positions to shoot more or less than would otherwise have been expected.

### Probability Distribution

Poisson provides a simple and interpretable method for converting expected shots into market probabilities, but historical player-shot counts show some evidence of overdispersion.

Alternative distributions and calibration methods remain candidates for future versions.

### Calibration

The current global calibration approach improves probability calibration but can assign a positive expectation to players whose raw expectation is zero.

Future development will test calibration methods that preserve sensible behaviour at the lower boundary and potentially account for different player roles.

### Wider Pricing Framework

Player shots form one component of a broader football pricing project.

Planned and developing markets include:

- Match results and goals
- Team shots
- Corners
- Player fouls
- Player cards
- Referee effects
- Match simulation
- Market-price comparison
- Model performance tracking

Each new component will follow the same principle:

**Add complexity only when historical testing demonstrates that it improves the model.**

## 8. Repository Scope

This notebook is a portfolio representation of a larger football analytics and pricing system.

Its purpose is to demonstrate:

- The modelling process
- Football reasoning behind feature selection
- Chronological backtesting
- Model calibration
- Rejection of unsuccessful features
- Translation of predictions into market probabilities
- Historical fake-live validation

To keep the public repository focused — and to retain parts of the underlying implementation — it does not include:

- The complete 8,500+ match historical database
- The full data-collection pipeline
- Production feature-engineering code
- Complete model parameters
- Automated live-fixture processing
- The full pricing engine
- Market-comparison and trading tools

Small examples and selected outputs are included to demonstrate the methodology without reproducing the complete working system.

---

### Project Status

**Player Shots V1 — Complete**

The current version includes:

**Team Shot Expectation → Player Shot Allocation → Expected Minutes → Calibration → Probability Distribution → Fair Market Prices**

Further models will be added to the wider project as they are developed and historically validated.